## 0) paths


In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd
import trackintel as ti
import duckdb

from eval import get_score

# Match path style from trackintel_hyperband_iteration_sim1.ipynb
IN_PARQUET = Path("../sim1_subset/data.zstd.parquet")
GT_PARQUET = Path("../sim1_subset/gt_sps.parquet")
SHARD_DIR = Path("../sim1_subset/shards_10")
OUT_DIR = Path("../outputs/sim1")

OUT_DIR.mkdir(parents=True, exist_ok=True)
SHARD_DIR.mkdir(parents=True, exist_ok=True)

print("IN_PARQUET =", IN_PARQUET.resolve())
print("GT_PARQUET =", GT_PARQUET.resolve())
print("SHARD_DIR  =", SHARD_DIR.resolve())
print("OUT_DIR    =", OUT_DIR.resolve())


IN_PARQUET = D:\staypoint_ptpe\sim1_subset\data.zstd.parquet
GT_PARQUET = D:\staypoint_ptpe\sim1_subset\gt_sps.parquet
SHARD_DIR  = D:\staypoint_ptpe\sim1_subset\shards_10
OUT_DIR    = D:\staypoint_ptpe\outputs\sim1


## 1) parameters


In [2]:
# Only run selected configs for now
PARAM_GROUPS = [
    {"name": "current", "DIST_M": 134.4, "TIME_MIN": 5.5, "GAP_MIN": 120.0},
    {"name": "baseline", "DIST_M": 50.0, "TIME_MIN": 5.0, "GAP_MIN": 25.0},
]

TIMEZONE = "UTC"
N_SHARDS = 10
SAVE_BEST_PARQUET = True  # set True to save best prediction parquet
RANK_METRIC = "f1"        # choose from precision / recall / f1 / f2

print("TIMEZONE =", TIMEZONE)
print("N_SHARDS =", N_SHARDS)
print("SAVE_BEST_PARQUET =", SAVE_BEST_PARQUET)


TIMEZONE = UTC
N_SHARDS = 10
SAVE_BEST_PARQUET = True


## 2) helpers


In [3]:
def ensure_tz(df, col, tz=TIMEZONE):
    s = pd.to_datetime(df[col], errors="coerce")
    if getattr(s.dt, "tz", None) is None:
        s = s.dt.tz_localize(tz)
    else:
        s = s.dt.tz_convert(tz)
    df[col] = s
    return df


def sp_to_eval_df(sp, tz=TIMEZONE):
    sp_out = sp.copy()

    sp_out["latitude"] = sp_out.geometry.y
    sp_out["longitude"] = sp_out.geometry.x
    sp_out["agent_id"] = sp_out["user_id"]

    time_candidates_start = ["started_at", "start_time", "arrive_time"]
    time_candidates_end = ["finished_at", "end_time", "leave_time"]

    start_col = next((c for c in time_candidates_start if c in sp_out.columns), None)
    end_col = next((c for c in time_candidates_end if c in sp_out.columns), None)

    if start_col is None or end_col is None:
        raise ValueError(f"Cannot find start/end columns in staypoints. Columns: {sp_out.columns.tolist()}")

    sp_out["arrive_time"] = pd.to_datetime(sp_out[start_col], errors="coerce")
    sp_out["leave_time"] = pd.to_datetime(sp_out[end_col], errors="coerce")

    sp_out = sp_out[["agent_id", "latitude", "longitude", "arrive_time", "leave_time"]].copy()
    sp_out = sp_out.dropna(subset=["agent_id", "latitude", "longitude", "arrive_time", "leave_time"])

    sp_out = ensure_tz(sp_out, "arrive_time", tz=tz)
    sp_out = ensure_tz(sp_out, "leave_time", tz=tz)
    return sp_out


def score_dict_to_flat(score_obj):
    out = {"precision": np.nan, "recall": np.nan, "f1": np.nan, "f2": np.nan}
    if isinstance(score_obj, dict):
        for k in out:
            if k in score_obj:
                out[k] = float(score_obj[k])
    return out


def build_sim1_shards_if_missing(n_shards=N_SHARDS):
    traj_shards = [SHARD_DIR / f"traj_shard_{i}.parquet" for i in range(1, n_shards + 1)]
    gt_shards = [SHARD_DIR / f"gt_shard_{i}.parquet" for i in range(1, n_shards + 1)]

    if all(p.exists() for p in traj_shards + gt_shards):
        print("All shard files already exist. Skip rebuilding.")
        return

    print("Shard files missing. Building", n_shards, "shards...")
    con = duckdb.connect()

    traj_users = con.execute(f"""
        SELECT DISTINCT agent AS user_id
        FROM read_parquet('{IN_PARQUET.as_posix()}')
    """).df()

    gt_users = con.execute(f"""
        SELECT DISTINCT user_id
        FROM read_parquet('{GT_PARQUET.as_posix()}')
    """).df()

    common_users = sorted(set(traj_users["user_id"]) & set(gt_users["user_id"]))
    print("Common users:", len(common_users))

    if len(common_users) == 0:
        raise ValueError("No common users found between trajectory and GT.")

    user_splits = np.array_split(common_users, n_shards)

    for i, users in enumerate(user_splits, start=1):
        users = [u for u in users.tolist() if pd.notna(u)]
        if not users:
            raise ValueError(f"Shard {i} has no users. Check n_shards.")

        users_df = pd.DataFrame({"user_id": users})
        con.register("users_df", users_df)

        traj_out = SHARD_DIR / f"traj_shard_{i}.parquet"
        gt_out = SHARD_DIR / f"gt_shard_{i}.parquet"

        con.execute(f"""
            COPY (
                SELECT agent, timestamp, longitude, latitude
                FROM read_parquet('{IN_PARQUET.as_posix()}')
                WHERE agent IN (SELECT user_id FROM users_df)
                ORDER BY agent, timestamp
            ) TO '{traj_out.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        con.execute(f"""
            COPY (
                SELECT user_id, started_at, finished_at, longitude, latitude
                FROM read_parquet('{GT_PARQUET.as_posix()}')
                WHERE user_id IN (SELECT user_id FROM users_df)
                ORDER BY user_id, started_at
            ) TO '{gt_out.as_posix()}' (FORMAT PARQUET, COMPRESSION ZSTD)
        """)

        print(f"Built shard {i}: users={len(users)}")

    con.close()


def load_one_traj_shard(shard_id):
    traj_file = SHARD_DIR / f"traj_shard_{shard_id}.parquet"
    df = pd.read_parquet(traj_file, columns=["agent", "timestamp", "longitude", "latitude"])
    df = df.rename(columns={"agent": "user_id", "timestamp": "tracked_at", "latitude": "n_lat", "longitude": "n_lon"})

    df["tracked_at"] = pd.to_datetime(df["tracked_at"], errors="coerce", utc=True)
    if TIMEZONE != "UTC":
        df["tracked_at"] = df["tracked_at"].dt.tz_convert(TIMEZONE)

    df = df.dropna(subset=["user_id", "tracked_at", "n_lat", "n_lon"])
    df = df.sort_values(["user_id", "tracked_at"])

    gdf = gpd.GeoDataFrame(
        df,
        geometry=gpd.points_from_xy(df["n_lon"], df["n_lat"]),
        crs="EPSG:4326",
    )

    pfs = ti.io.read_positionfixes_gpd(
        gdf,
        tracked_at="tracked_at",
        user_id="user_id",
        tz=TIMEZONE,
    )
    return pfs


def load_one_gt_shard(shard_id):
    gt_file = SHARD_DIR / f"gt_shard_{shard_id}.parquet"
    gt = pd.read_parquet(gt_file, columns=["user_id", "started_at", "finished_at", "longitude", "latitude"]).rename(columns={
        "user_id": "agent_id",
        "started_at": "arrive_time",
        "finished_at": "leave_time",
    })

    gt = gt[["agent_id", "latitude", "longitude", "arrive_time", "leave_time"]].copy()
    gt = ensure_tz(gt, "arrive_time", tz=TIMEZONE)
    gt = ensure_tz(gt, "leave_time", tz=TIMEZONE)
    gt = gt.dropna(subset=["agent_id", "latitude", "longitude", "arrive_time", "leave_time"])
    return gt


def evaluate_config_on_one_shard(cfg, shard_id):
    pfs = load_one_traj_shard(shard_id)
    gt = load_one_gt_shard(shard_id)

    _, sp = pfs.generate_staypoints(
        method="sliding",
        distance_metric="haversine",
        dist_threshold=cfg["DIST_M"],
        time_threshold=cfg["TIME_MIN"],
        gap_threshold=cfg["GAP_MIN"],
        exclude_duplicate_pfs=True,
    )

    pred_eval = sp_to_eval_df(sp, tz=TIMEZONE)
    score_obj = get_score(ground_truth_df=gt, calculated_df=pred_eval)

    out = {
        "shard_id": int(shard_id),
        "n_gt": int(len(gt)),
        "n_pred": int(len(pred_eval)),
        "score": score_dict_to_flat(score_obj),
    }

    del pfs, sp, gt, pred_eval
    gc.collect()
    return out


def combine_scores_from_shards(shard_results):
    total_gt = 0
    total_pred = 0
    total_matched_gt = 0.0
    total_matched_pred = 0.0

    for item in shard_results:
        n_gt = int(item["n_gt"])
        n_pred = int(item["n_pred"])
        precision_i = float(item["score"].get("precision", np.nan))
        recall_i = float(item["score"].get("recall", np.nan))

        matched_gt_i = 0.0 if n_gt == 0 else recall_i * n_gt
        matched_pred_i = 0.0 if n_pred == 0 else precision_i * n_pred

        total_gt += n_gt
        total_pred += n_pred
        total_matched_gt += matched_gt_i
        total_matched_pred += matched_pred_i

    precision = 0.0 if total_pred == 0 else (total_matched_pred / total_pred)
    recall = 0.0 if total_gt == 0 else (total_matched_gt / total_gt)

    f1 = 0.0 if (precision + recall) == 0 else (2 * precision * recall / (precision + recall))

    beta2 = 2.0
    f2 = 0.0 if ((beta2**2) * precision + recall) == 0 else (
        (1 + beta2**2) * precision * recall / ((beta2**2) * precision + recall)
    )

    return {
        "precision": float(precision),
        "recall": float(recall),
        "f1": float(f1),
        "f2": float(f2),
        "n_gt": int(total_gt),
        "n_pred": int(total_pred),
    }


def evaluate_config(cfg):
    shard_results = []
    for shard_id in range(1, N_SHARDS + 1):
        shard_results.append(evaluate_config_on_one_shard(cfg, shard_id))

    combined = combine_scores_from_shards(shard_results)
    return combined, shard_results


def predict_config_across_shards(cfg):
    preds = []
    for shard_id in range(1, N_SHARDS + 1):
        pfs = load_one_traj_shard(shard_id)

        _, sp = pfs.generate_staypoints(
            method="sliding",
            distance_metric="haversine",
            dist_threshold=cfg["DIST_M"],
            time_threshold=cfg["TIME_MIN"],
            gap_threshold=cfg["GAP_MIN"],
            exclude_duplicate_pfs=True,
        )

        pred_eval = sp_to_eval_df(sp, tz=TIMEZONE)
        pred_eval["shard_id"] = int(shard_id)
        preds.append(pred_eval)

        del pfs, sp, pred_eval
        gc.collect()

    if not preds:
        return pd.DataFrame(columns=["agent_id", "latitude", "longitude", "arrive_time", "leave_time", "shard_id"])

    return pd.concat(preds, ignore_index=True)


## 3) load noisy trajectory


In [4]:
build_sim1_shards_if_missing(n_shards=N_SHARDS)

missing = []
for i in range(1, N_SHARDS + 1):
    t = SHARD_DIR / f"traj_shard_{i}.parquet"
    g = SHARD_DIR / f"gt_shard_{i}.parquet"
    if (not t.exists()) or (not g.exists()):
        missing.append(i)

if missing:
    raise FileNotFoundError(f"Missing shard files for shard ids: {missing}")

print("Shard check passed for", N_SHARDS, "shards")


All shard files already exist. Skip rebuilding.
Shard check passed for 10 shards


## 4) load ground truth


In [5]:
# Optional sanity check: total GT rows across shards
gt_total_rows = 0
for shard_id in range(1, N_SHARDS + 1):
    gt_shard = load_one_gt_shard(shard_id)
    gt_total_rows += len(gt_shard)

print("Total GT rows across shards:", gt_total_rows)


Total GT rows across shards: 167660


## 5) generate staypoints + score for each parameter group


In [6]:
all_results = []

for params in PARAM_GROUPS:
    print("\n" + "=" * 60)
    print(f"Run: {params['name']}")
    print(f"DIST_M={params['DIST_M']}, TIME_MIN={params['TIME_MIN']}, GAP_MIN={params['GAP_MIN']}")

    combined_score, shard_results = evaluate_config(params)

    print("Combined score:", {k: combined_score[k] for k in ["precision", "recall", "f1", "f2"]})
    print("Pred rows:", combined_score["n_pred"])
    print("GT rows  :", combined_score["n_gt"])

    row = {
        "name": params["name"],
        "DIST_M": params["DIST_M"],
        "TIME_MIN": params["TIME_MIN"],
        "GAP_MIN": params["GAP_MIN"],
        "n_pred": combined_score["n_pred"],
        "n_gt": combined_score["n_gt"],
        "precision": combined_score["precision"],
        "recall": combined_score["recall"],
        "f1": combined_score["f1"],
        "f2": combined_score["f2"],
    }
    all_results.append(row)



Run: current
DIST_M=134.4, TIME_MIN=5.5, GAP_MIN=120.0
[2026-03-22 14:41:17]: Calculating recall score...
[2026-03-22 14:41:27]: Calculating precision score...
[2026-03-22 14:43:29]: Calculating recall score...
[2026-03-22 14:43:39]: Calculating precision score...
[2026-03-22 14:45:38]: Calculating recall score...
[2026-03-22 14:45:47]: Calculating precision score...
[2026-03-22 14:47:53]: Calculating recall score...
[2026-03-22 14:48:03]: Calculating precision score...
[2026-03-22 14:50:09]: Calculating recall score...
[2026-03-22 14:50:19]: Calculating precision score...
[2026-03-22 14:52:29]: Calculating recall score...
[2026-03-22 14:52:39]: Calculating precision score...
[2026-03-22 14:54:52]: Calculating recall score...
[2026-03-22 14:55:01]: Calculating precision score...
[2026-03-22 14:57:12]: Calculating recall score...
[2026-03-22 14:57:22]: Calculating precision score...
[2026-03-22 14:59:34]: Calculating recall score...
[2026-03-22 14:59:43]: Calculating precision score...

## 6) final summary


In [7]:
summary_df = pd.DataFrame(all_results)
print("\n" + "=" * 60)
print("Final summary:")
print(summary_df)

if RANK_METRIC in summary_df.columns:
    best_idx = summary_df[RANK_METRIC].astype(float).idxmax()
else:
    best_idx = summary_df["f1"].astype(float).idxmax() if "f1" in summary_df.columns else summary_df.index[0]

best_row = summary_df.loc[best_idx]
best_cfg = {
    "name": best_row["name"],
    "DIST_M": float(best_row["DIST_M"]),
    "TIME_MIN": float(best_row["TIME_MIN"]),
    "GAP_MIN": float(best_row["GAP_MIN"]),
}

print("\nBest config by", RANK_METRIC, ":")
print({
    "name": best_cfg["name"],
    "dist_m": best_cfg["DIST_M"],
    "time_min": best_cfg["TIME_MIN"],
    "gap_min": best_cfg["GAP_MIN"],
})

if SAVE_BEST_PARQUET:
    best_pred = predict_config_across_shards(best_cfg)
    out_path = OUT_DIR / f"sim1_best_dist{best_cfg['DIST_M']}_time{best_cfg['TIME_MIN']}_gap{best_cfg['GAP_MIN']}.parquet"
    best_pred.to_parquet(out_path, index=False)
    print("Saved best parquet:", out_path.resolve())
    print("Saved rows:", len(best_pred))
else:
    print("SAVE_BEST_PARQUET=False -> skip saving best parquet")



Final summary:
       name  DIST_M  TIME_MIN  GAP_MIN  n_pred    n_gt  precision    recall  \
0   current   134.4       5.5    120.0  143213  167660   0.496889  0.424424   
1  baseline    50.0       5.0     25.0  565817  167660   0.074188  0.250328   

         f1        f2  
0  0.457807  0.437176  
1  0.114456  0.169732  

Best config by f1 :
{'name': 'current', 'dist_m': 134.4, 'time_min': 5.5, 'gap_min': 120.0}
Saved best parquet: D:\staypoint_ptpe\outputs\sim1\sim1_best_dist134.4_time5.5_gap120.0.parquet
Saved rows: 143213
